# 14_msd_family_distance.ipynb

Task 5.7: đo **MSD** (Mintun et al., R33) giữa mỗi họ suy giảm và hai bộ augmentation (TrivialAugment Wide, augmentation suy giảm của đối chứng 4.5) bằng Swin-Tiny reference, rồi chia nhóm gần/xa bằng số thay cho chia theo tên. Sau đó hoàn tất task 5.8: chia họ thành train, val, test và **khóa** `configs/unseen_families_v2.yaml`. Chỉ suy luận, **không mở tập test**.

Điều kiện chạy: `reports/unseen_v2/calibration.json` từ notebook 13 đã nằm trên nhánh.

Quy tắc đăng ký trước (`registration.msd`, `registration.roles`):

1. 100 ảnh validation chọn bằng hoán vị seed cố định, 1000 mẫu mỗi bộ augmentation, đặc trưng là lớp ẩn cuối của Swin-Tiny (lật và xoay, không có augmentation nào khác).
2. Ngưỡng gần/xa = MSD leave-one-out lớn nhất trong 14 họ đã thấy. Họ có khoảng bootstrap 95% chứa ngưỡng xếp là `ambiguous`.
3. Chia vai trò theo tầng nhóm MSD, thứ tự theo sha256(seed:tên họ), họ test chỉ từ nguồn ngoài.

## 1. Môi trường

Internet On, Accelerator GPU, cùng dataset input như notebook 13. Họ ngoài chạy trên CPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "unseen-families-v2-5.5-5.7-5.8"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

In [ ]:
import subprocess
import sys

# imagecorruptions 1.1.2 import pkg_resources (setuptools dưới 81). Một số họ lỗi với numpy mới, probe_families ở dưới sẽ loại.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecorruptions==1.1.2", "setuptools<81"], check=True)
import numpy, scipy, skimage, torch, torchvision, timm
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| scikit-image", skimage.__version__)
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| timm", timm.__version__)

In [ ]:
import json

import pandas as pd
import torch
import yaml

from src.corruption import load_conditions, load_seed
from src.corruption.benchmark import load_config
from src.evaluation import unseen_families_v2 as U
from src.evaluation.reference import collect_checkpoint, load_trained_model
from src.training.loop import resolve_device

CONFIG = U.CONFIG_PATH
if load_config(CONFIG)["locked"]:
    raise RuntimeError("configs/unseen_families_v2.yaml đã khóa. Không chạy lại: chạy lại có thể ghi đè bộ đã khóa bằng kết quả khác.")
OUT_DIR = REPO_DIR / "reports/unseen_v2"
registration = U.load_registration(CONFIG)
msd_cfg = registration["msd"]
pipeline_cfg = yaml.safe_load((REPO_DIR / "configs/pipeline.yaml").read_text(encoding="utf-8"))
calibration_path = OUT_DIR / "calibration.json"
if not calibration_path.exists():
    raise FileNotFoundError("Thiếu reports/unseen_v2/calibration.json. Chạy notebook 13, commit file lên nhánh rồi chạy lại cell này.")
calibration = json.loads(calibration_path.read_text(encoding="utf-8"))
kept = calibration["kept"]
print(f"{len(kept)} họ đã khớp độ khó: {', '.join(sorted(kept))}")

## 2. Mô hình tham chiếu và ảnh

MSD đo trên Swin-Tiny reference, huấn luyện chỉ với lật và xoay, như yêu cầu của phương pháp.

In [ ]:
from src.evaluation.msd import Featurizer, select_images

device = resolve_device()
checkpoint = collect_checkpoint("/kaggle/input", registration["reference"]["checkpoint"], Path("/kaggle/working/checkpoints"))
model = load_trained_model(yaml.safe_load((REPO_DIR / registration["reference"]["train_config"]).read_text(encoding="utf-8")), checkpoint, len(pipeline_cfg["classes"]), device)
featurizer = Featurizer(model, pipeline_cfg["mean"], pipeline_cfg["std"], device, batch_size=int(pipeline_cfg["batch_size"]))
images = select_images(pipeline_cfg, msd_cfg["split"], msd_cfg["n_images"], msd_cfg["seed"])
print(f"{registration['reference']['model']}, {images.shape[0]} ảnh {msd_cfg['split']}, đặc trưng {featurizer.features(images[:2]).shape[1]} chiều")

## 3. Hai bộ augmentation

1000 mẫu mỗi bộ, mỗi mẫu là một (phép, cường độ) cố định áp lên cả 100 ảnh. Mất vài phút đến vài chục phút. Kết quả lưu vào `/kaggle/working/msd_cache` để chạy lại không phải đo lại.

In [ ]:
import time

from src.evaluation.msd import AugmentationPool, build_degradation_pool, build_trivial_augment_pool

CACHE = Path("/kaggle/working/msd_cache")
CACHE.mkdir(exist_ok=True)
seen_conditions = load_conditions(registration["seen_benchmark"], "val")
seen_seed = load_seed(registration["seen_benchmark"])

def cached(name, builder):
    path = CACHE / f"{name}.pt"
    if path.exists():
        saved = torch.load(path)
        return AugmentationPool(saved["name"], saved["deltas"], saved["labels"], saved["meta"])
    start = time.time()
    pool = builder()
    torch.save({"name": pool.name, "deltas": pool.deltas, "labels": pool.labels, "meta": pool.meta}, path)
    print(f"{name}: {time.time() - start:.0f} giây")
    return pool

degradation_pool = cached("degradation_aug", lambda: build_degradation_pool(featurizer, images, seen_conditions, msd_cfg["n_samples"], msd_cfg["seed"], msd_cfg["jitter"]))
ta_pool = cached("trivialaugment_wide", lambda: build_trivial_augment_pool(featurizer, images, msd_cfg["n_samples"], msd_cfg["seed"]))
print(degradation_pool.deltas.shape, ta_pool.deltas.shape)

## 4. Ngưỡng gần/xa từ 14 họ đã thấy

MSD leave-one-out: mỗi họ đã thấy đo tới augmentation suy giảm đã bỏ chính họ đó. Ngưỡng là giá trị lớn nhất.

In [ ]:
from src.evaluation.msd import family_deltas, leave_one_out_msd, msd, near_far_threshold

seen_families = list(dict.fromkeys(spec["family"] for spec in seen_conditions.values()))
seen_deltas = {family: family_deltas(featurizer, images, seen_conditions, family, seen_seed) for family in seen_families}
loo = leave_one_out_msd(degradation_pool, seen_deltas, msd_cfg["n_bootstrap"], msd_cfg["seed"])
threshold = near_far_threshold(loo)
display(pd.DataFrame({f: r.as_dict() for f, r in loo.items()}).T.round(4).sort_values("msd"))
print(f"Ngưỡng gần/xa (max leave-one-out): {threshold:.4f}")

## 5. MSD của các họ mới

Mỗi họ đã khớp độ khó đo tới augmentation suy giảm (đủ 14 họ) và tới TrivialAugment. Nhóm gần/xa xếp theo ngưỡng ở trên, `ambiguous` nếu khoảng bootstrap chứa ngưỡng.

In [ ]:
from src.evaluation.msd import classify_near_far

new_conditions = U.matched_conditions(kept)
new_families = sorted(kept)
new_deltas = {family: family_deltas(featurizer, images, new_conditions, family, seen_seed) for family in new_families}
rows = []
for family, delta in new_deltas.items():
    to_degradation = msd(degradation_pool, delta, n_bootstrap=msd_cfg["n_bootstrap"], seed=msd_cfg["seed"])
    to_ta = msd(ta_pool, delta, n_bootstrap=msd_cfg["n_bootstrap"], seed=msd_cfg["seed"])
    rows.append({
        "family": family, "group": classify_near_far(to_degradation, threshold),
        "msd": to_degradation.value, "ci_low": to_degradation.ci_low, "ci_high": to_degradation.ci_high, "nearest": to_degradation.nearest,
        "msd_ta": to_ta.value, "ci_low_ta": to_ta.ci_low, "ci_high_ta": to_ta.ci_high, "nearest_ta": to_ta.nearest,
    })
new_table = pd.DataFrame(rows).set_index("family").sort_values("msd")
display(new_table.round(4))
print(new_table["group"].value_counts().to_dict())

## 6. Bộ v1 và 14 họ đã thấy, đo cùng thước

Bộ v1 chia gần/xa theo tên (`configs/unseen_corruption.yaml`). Ở đây đo cùng ngưỡng để thấy chia theo tên có khớp chia theo số không. Họ đã thấy đo với TrivialAugment để biết TA phủ họ nào.

In [ ]:
v1_conditions = load_conditions("configs/unseen_corruption.yaml", "val")
v1_seed = load_seed("configs/unseen_corruption.yaml")
v1_rows = []
for family in dict.fromkeys(spec["family"] for spec in v1_conditions.values()):
    delta = family_deltas(featurizer, images, v1_conditions, family, v1_seed)
    result = msd(degradation_pool, delta, n_bootstrap=msd_cfg["n_bootstrap"], seed=msd_cfg["seed"])
    v1_rows.append({"family": family, "group_by_name": next(s["group"] for s in v1_conditions.values() if s["family"] == family), "group_by_msd": classify_near_far(result, threshold), **result.as_dict()})
v1_table = pd.DataFrame(v1_rows).set_index("family")
display(v1_table.round(4))

seen_ta = {family: msd(ta_pool, delta, n_bootstrap=msd_cfg["n_bootstrap"], seed=msd_cfg["seed"]).as_dict() for family, delta in seen_deltas.items()}
seen_ta_table = pd.DataFrame(seen_ta).T.sort_values("msd")
display(seen_ta_table.round(4))

## 7. Chồng lấn của TrivialAugment với họ có cùng quy ước hệ số

Đếm bằng cách liệt kê: xác suất một lần gọi TA cho hệ số trong ±10% quanh mức của họ.

In [ ]:
from src.evaluation.ta_overlap import overlap_table

overlap = pd.DataFrame(overlap_table({**seen_conditions, **v1_conditions}))
display(overlap.round(4))

## 8. Khoảng cách giữa các họ đã thấy

Ma trận ‖f(A) − f(B)‖ giữa 14 họ đã thấy, để chọn nhóm họ bỏ ở task 5.10 theo số (họ nào gần nhau thì cùng nhóm) thay vì theo tên.

In [ ]:
from IPython.display import Image

from src.evaluation.msd import pairwise_distances
from src.evaluation.msd_plots import plot_distance_matrix, plot_msd_bars

matrix = pairwise_distances(seen_deltas)
matrix_df = pd.DataFrame(matrix)
matrix_df.to_csv(OUT_DIR / "seen_pairwise_distance.csv")
display(Image(filename=plot_distance_matrix(matrix, OUT_DIR / "seen_pairwise_distance.png", "Khoảng cách đặc trưng giữa 14 họ đã thấy")))
bar_rows = [{"family": f, "msd": r.value, "ci_low": r.ci_low, "ci_high": r.ci_high, "group": "seen"} for f, r in loo.items()] + new_table.reset_index().to_dict("records")
display(Image(filename=plot_msd_bars(bar_rows, threshold, OUT_DIR / "msd_to_degradation_aug.png", "MSD tới augmentation suy giảm: họ đã thấy (leave-one-out) và họ mới")))

## 9. Vai trò train, val, test

Theo `family_roles.assign_roles`: phân tầng theo nhóm, sắp xếp bằng sha256, họ test chỉ từ nguồn ngoài.

In [ ]:
from src.corruption.family_roles import assign_roles

groups = new_table["group"].to_dict()
roles = assign_roles({family: {"group": groups[family], "source": U.family_sources([family])[family]} for family in new_families}, load_seed(CONFIG), registration["roles"]["fractions"])
role_table = pd.DataFrame({"group": groups, "role": roles}).sort_values(["group", "role"])
display(role_table)
display(pd.crosstab(role_table["group"], role_table["role"]))

## 10. Ghi kết quả và khóa

`msd.json` ghi trước, mã sha256 của nó và của `calibration.json` ghi vào `lock.inputs`. Sau cell này `configs/unseen_families_v2.yaml` có `locked: true`. Commit file này cùng `reports/unseen_v2/`. Từ đây không chỉnh mức, nhóm, vai trò.

In [ ]:
import hashlib

msd_record = {
    "n_images": msd_cfg["n_images"], "n_samples": msd_cfg["n_samples"], "n_bootstrap": msd_cfg["n_bootstrap"], "threshold": threshold, "threshold_rule": msd_cfg["threshold_rule"],
    "seen_leave_one_out": {f: r.as_dict() for f, r in loo.items()},
    "seen_to_trivialaugment": seen_ta,
    "new_families": new_table.reset_index().to_dict("records"),
    "v1_families": v1_table.reset_index().to_dict("records"),
    "ta_overlap": overlap.to_dict("records"),
    "roles": roles,
    "seen_pairwise_distance": matrix,
}
msd_path = OUT_DIR / "msd.json"
msd_path.write_text(json.dumps(msd_record, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
new_table.to_csv(OUT_DIR / "msd_new_families.csv")
v1_table.to_csv(OUT_DIR / "msd_v1_families.csv")
seen_ta_table.to_csv(OUT_DIR / "msd_seen_to_trivialaugment.csv")
role_table.to_csv(OUT_DIR / "roles.csv")

cfg = U.assemble_config(
    registration, kept, groups, roles, load_seed(CONFIG),
    inputs={"calibration": U.file_sha256(calibration_path), "msd": U.file_sha256(msd_path)},
)
path = U.dump_config(cfg, CONFIG)
print(f"Đã ghi {path.relative_to(REPO_DIR)}  sha256 {cfg['lock']['sha256']}")

# kiểm tra đọc lại: khớp mã băm, đúng số điều kiện, mở val không cần biến môi trường, mở test thì bị chặn
for key in ("FINAL_FAMILY_EVAL",):
    os.environ.pop(key, None)
reloaded = load_conditions(CONFIG, "val")
assert len(reloaded) == 3 * len(kept)
try:
    load_conditions(CONFIG, "test")
    raise AssertionError("Tập test không được mở khi thiếu FINAL_FAMILY_EVAL=1.")
except PermissionError as error:
    print("Cổng test hoạt động:", error)

## 11. Nén kết quả

Tải `unseen_v2_msd_outputs.zip` về, giải nén vào repo (giữ nguyên đường dẫn), kiểm tra `git diff configs/unseen_families_v2.yaml`, rồi commit.

In [ ]:
import zipfile

archive = Path("/kaggle/working/unseen_v2_msd_outputs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT_DIR.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(REPO_DIR).as_posix())
    bundle.write(REPO_DIR / CONFIG, CONFIG)
print(f"Đã nén {archive} ({archive.stat().st_size / 1024**2:.1f} MB)")